#PREPROCESSING

In [ ]:
import pandas as pd
from datasets import Dataset

# Load your CSV data
csv_file = "dataset_finetune_s.csv"  # Replace with the path to your file
df = pd.read_csv(csv_file)

# Check the first few rows to confirm the structure
print(df.head())

In [ ]:
# Convert DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Create a function to format the data as input-output pairs
def preprocess_data(example):
    input_text = example["Question"]
    target_text = example["Answer"]
    return {"input": input_text, "output": target_text}

# Apply the preprocessing function to the dataset
tokenized_dataset = dataset.map(preprocess_data, batched=True)

# Display tokenized dataset to confirm preprocessing
print(tokenized_dataset[0])

#TOKENIZATION

In [ ]:
from transformers import AutoTokenizer

# Load tokenizer for FLAN-T5
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def tokenize_function(example):
    # Tokenize inputs (questions) and outputs (answers)
    input_enc = tokenizer(example['input'], truncation=True, padding="max_length", max_length=1024)
    output_enc = tokenizer(example['output'], truncation=True, padding="max_length", max_length=400)
    
    # Set the output as labels
    output_enc["labels"] = output_enc["input_ids"]  # Correctly setting the labels for decoder
    return {**input_enc, **output_enc}

# Tokenize the dataset
tokenized_dataset = tokenized_dataset.map(tokenize_function, batched=True)

# Show tokenized example
print(tokenized_dataset[0])

#FINE TUNING THE MODEL

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Load the pre-trained FLAN-T5 model
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Split the dataset into train and eval sets (80% train, 20% eval)
train_dataset = tokenized_dataset.shuffle(seed=42).select([i for i in range(int(0.8 * len(tokenized_dataset)))])
eval_dataset = tokenized_dataset.shuffle(seed=42).select([i for i in range(int(0.8 * len(tokenized_dataset)), len(tokenized_dataset))])

# Define the training arguments
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./flan-t5-finetuned",
#     evaluation_strategy="steps",          
#     eval_steps=500,                       
#     learning_rate=3e-5,
#     per_device_train_batch_size=4,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     save_total_limit=1,
#     # predict_with_generate=True,
#     fp16=True
# )
training_args = Seq2SeqTrainingArguments(
        output_dir="./flan-t5-base-finetuned",  # New save path for the model
        evaluation_strategy="steps",
        save_strategy="steps",
        eval_steps=100,
        save_steps=100,
        logging_dir="./logs",
        logging_steps=50,
        learning_rate=5e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        max_steps=100,
        weight_decay=0.01,
        save_total_limit=3,
        predict_with_generate=True,
        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        push_to_hub=False,
    )

# Define the trainer
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     tokenizer=tokenizer
# )
trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        # data_collator=data_collator
    )

# Start fine-tuning the model
checkpoint_path = "flan-t5-base-finetuned/checkpoint-100"
trainer.train(resume_from_checkpoint=checkpoint_path)

In [109]:
trainer.save_model("./final_finetuned-base")

#EVALUATION

In [ ]:
# Evaluate the model
results = trainer.evaluate(eval_dataset)

# Print evaluation results
print(results)

# Test a few examples
sample_input = "List the risk factors for heart attack in comma separated values."
generated_output = trainer.predict(eval_dataset)

print(generated_output)


#TESTING

In [116]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Load your dataset
df = pd.read_csv("dataset_finetune_s.csv")

# Split the dataset into training (80%) and testing (20%)
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)

# Save evaluation dataset if needed
# eval_df.to_csv("evaluation_dataset.csv", index=False)

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load the fine-tuned model and tokenizer
model_path = "new_finetuned-base"  # Path to the saved model
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
# model = AutoModelForSeq2SeqLM.from_pretrained("./flan-t5-base-finetuned/checkpoint-1000")
# model = AutoModelForSeq2SeqLM.from_pretrained("./flan-t5-finetuned/checkpoint-600")
# tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

In [9]:
# Define your custom context and question
context = "what's bothering you? i've had a sore throat and trouble swallowing for a few days. have you had a fever? yes, a mild one. it sounds like a throat infection. i'll prescribe antibiotics and a pain reliever. drink warm fluids and rest. alright, thank you.  you; good morning. good morning. i've been having a persistent headache for the past week. i see. any other symptoms? sometimes i feel dizzy. have you been sleeping well and staying hydrated? not really.  drink plenty of water and get rest. thank you."
question = "List the patient's chief complaints in comma separated values."

In [10]:
# Combine context and question
input_text = f"Context: {context} Question: {question} Answer:"

inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=1024, truncation=True)

In [11]:
with torch.no_grad():
    outputs = model.generate(inputs, max_length=400, num_beams=5, early_stopping=True)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# Print the result
print(answer)

sore throat
